# 01 — Quickstart: what this machine actually runs

AutosXtract extracts text from PDFs by descending steps from the cheapest to the
most expensive and stopping at the first one that produces acceptable text. That
design only pays because the distribution is uneven: measured across two real
archives, **31% of documents are resolved by the native text layer at 13.4 ms**,
and only the remainder pays the ~500 ms of OCR.

This notebook is the shortest honest path to a `Result`. Four things happen in
it, and the third is the one people skip:

1. check the install;
2. **ask the machine which cascade it has** — the answer is different on macOS
   and on Linux, and a cascade one step short does not announce itself;
3. build a document (synthetic — see below);
4. read the result, and then read its *provenance*, which is the library's
   product as much as the text is.

### The rule that governs every notebook here

No real document enters this repository (CLAUDE.md §9), and that includes the
notebooks. Every PDF below is generated in-cell by PyMuPDF with invented text,
which is the same technique `tests/conftest.py` uses. The case number that
appears in the sample text has a **deliberately invalid check digit**, so
`scripts/privacy_check.py` — which validates Brazilian tax IDs, company IDs and
case numbers arithmetically — stays quiet and nobody has to decide, file by
file, whether a number belongs to somebody.

In [ ]:
import logging

# The OCR backend chatters at INFO while it loads its ONNX models. That noise is
# the vendor's, not the library's, and it drowns the two lines this notebook is
# actually about.
logging.disable(logging.INFO)

import autosxtract
from autosxtract import Cascade, Config, diagnose, engine_order, is_apple, vision_available

print("autosxtract", autosxtract.__version__)
print("apple hardware:", is_apple(), "| vision:", vision_available())

## Which cascade does *this* machine have?

The library has one rule about missing tools, and it is the reason `diagnose`
exists rather than an exception: **a missing engine is never an exception**
(CLAUDE.md §3). `available()` returns `(False, reason)` in words, the step goes
inert, and the cascade moves on — because *the absence of a tool is not evidence
about the document*. Treating "I have no OCR" as "the page is empty" switches
the pipeline off in silence.

The corollary is the half people forget: degrading without breaking is right,
degrading **without warning** is not. So every reason lands in two places — the
provenance of each result, and the table below.

On macOS the OCR chain is `vision -> paddle`; everywhere else Vision simply is
not there and PP-OCRv6 tiny becomes step 2. Nothing else about the cascade
changes with the platform: same steps, same gates, same contest.

In [ ]:
for name, ok, reason in diagnose():
    print(f"  [{'x' if ok else ' '}] {name:<12} {reason}")

print()
print("OCR engines chosen here :", engine_order() or "(none)")
print("assembled cascade       :", " -> ".join(Cascade().names))

Two absences in that table are worth naming, because they change what the
later notebooks can show on a Linux machine:

- **`vision` / `ocrmac`** are Apple-only. Notebook 03's containment layers and
  notebook 04's two-engine comparison lose a participant here, not their point.
- **`tesseract`** is the *witness* for vetoes 3 to 5 (notebook 06). It is
  deliberately kept out of the transcription chain — an engine that both
  produces a candidate and vouches for the others is not independent evidence.
  Without it installed, `LocalReading` is `None`, which means **"I don't know"**
  and skips those three vetoes. It never becomes "there is no text".

The equivalent from a terminal is `autosxtract diagnose`, and it also prints the
resolved parallelism and whether the model weights are on disk.

## A document, invented here

`insert_textbox` writes a real text layer, so this PDF is the *digital filing*
case: the native step will find text without any OCR being loaded. The text sits
in the top half of the sheet only, which is the real arrangement of a filing
with a scanned attachment underneath — notebook 03 uses the bottom half.

In [ ]:
import pymupdf

BODY = (
    "EXCELENTISSIMO SENHOR DOUTOR JUIZ DE DIREITO DA VARA CIVEL\n\n"
    "O requerente, nos autos do processo 0001234-56.2020.8.12.0001, vem "
    "respeitosamente a presenca de Vossa Excelencia expor e ao final requerer "
    "o que segue. A decisao proferida nestes autos determinou a intimacao da "
    "parte executada para que se manifestasse no prazo legal, sob pena de "
    "preclusao. O exequente informa que a diligencia foi cumprida e que o "
    "mandado retornou devidamente cumprido em 17/03/2005, conforme certidao "
    "de folhas seguintes.\n\n"
    "Nestes termos, pede deferimento."
)


def digital_pdf(pages: list[str]) -> bytes:
    """A PDF with a real text layer. Nothing here comes off a disk."""
    doc = pymupdf.open()
    for text in pages:
        page = doc.new_page()
        if text:
            page.insert_textbox(pymupdf.Rect(50, 50, 550, 380), text, fontsize=11)
    data = doc.tobytes()
    doc.close()
    return data


filing = digital_pdf([BODY, BODY])
print(len(filing), "bytes,", "2 pages")

## Extracting

`Cascade` is meant to be built **once** and called many times: engines load once
per process, and instantiating per document throws away that load cache — which
is the dominant cost of the first call.

`extract` never raises because of a missing engine or an unreadable PDF. It
returns a `Result` whose provenance says what happened at each step. Extraction
is a process with an uncertain outcome, and an explained uncertain outcome is
worth more than a traceback.

In [ ]:
cascade = Cascade()
result = cascade.extract(filing, identifier="filing.pdf")

print("step   :", result.step)
print("score  :", result.score)
print("chars  :", len(result.text))
print("ms     :", round(result.ms, 1))
print("empty  :", result.empty)
print()
print(result.text[:220], "...")

## Provenance: the part that makes the answer auditable

"The system extracted the text" is not an auditable answer. "The native step read
41 characters and was refused on density, and PP-OCRv6 read 3,812 and passed" is.
That sentence is `Result.provenance`, and the structured version is
`Result.attempts` — one `Attempt` per step, each carrying `accepted`, the
`reason`, the characters it produced and how long it took.

The interesting row is always `accepted == False` **with characters** in it: that
is a step whose text was refused as a stopping point but which is still competing
for the document's text slot (CLAUDE.md §5). Notebook 02 is entirely about that
distinction.

In [ ]:
def attempts_table(result) -> None:
    print(f"{'step':<22} {'ok':<4} {'chars':>7} {'ms':>8}  reason")
    print("-" * 78)
    for a in result.attempts:
        print(f"{a.step:<22} {str(a.accepted):<4} {a.chars:>7} {a.ms:>8.1f}  {a.reason}")


print(result.provenance)
print()
attempts_table(result)

`to_dict()` is the serialisable form — what goes to JSON, a log or a database.
It flattens `details` into the top level, so whatever a step chose to record
(engine confidence, which pages were sent to OCR, the containment report) travels
with the text instead of being lost at the boundary.

In [ ]:
import json

payload = result.to_dict()
print(sorted(payload))
print()

# The text itself is the one field worth truncating in a log.
compact = dict(payload, text=payload["text"][:60] + " ...")
print(json.dumps(compact, indent=2, ensure_ascii=False)[:1200])

## The other branch: a page with no text layer

The same generator, one step further: render the digital page to pixels and put
*that* into a new PDF. There is no text layer left, so the native step has
nothing to read and the cascade descends to OCR. This is the fixture that makes
the acceptance gate do real work — a genuinely blank sheet would be accepted for
want of an alternative rather than on its merits.

The cell degrades on purpose: with no OCR engine installed it reports the
provenance and moves on, which is exactly what the library does.

In [ ]:
def scanned_pdf(pages: list[str], dpi: int = 180) -> bytes:
    """The same pages, flattened to pixels: ink, no text layer."""
    source = pymupdf.open("pdf", digital_pdf(pages))
    out = pymupdf.open()
    for i in range(len(source)):
        pixmap = source[i].get_pixmap(dpi=dpi, colorspace=pymupdf.csGRAY)
        page = out.new_page(width=source[i].rect.width, height=source[i].rect.height)
        page.insert_image(page.rect, stream=pixmap.tobytes("png"))
    data = out.tobytes()
    out.close()
    source.close()
    return data


scanned = scanned_pdf([BODY])

if not engine_order():
    print("no OCR engine on this machine — the cascade will refuse and say why.")

ocr_result = cascade.extract(scanned, identifier="scanned.pdf")
print(ocr_result.provenance)
print()
attempts_table(ocr_result)
print()
print(repr(ocr_result.text[:200]))

If an engine ran, compare its output against `BODY` above rather than admiring
the score: the reading is usually very good and still not identical. That gap is
the whole subject of notebooks 03 and 04 — the score describes the text that came
out, never the fraction of the page left behind.

## Where to go next

| notebook | what it teaches |
|---|---|
| **02** | each step run by hand, and why a *refused* step keeps competing |
| **03** | the two gates, the score, and why the stamp is stripped before measuring |
| **04** | the `Config` fields that matter, and how to measure a change instead of guessing |
| **05** | writing an `Engine` against the Protocol alone |
| **06** | writing a `Step`, and what `expensive = True` costs you |
| **07** | a pattern pack for another language or domain |